# P1_03 · Kvasir-SEG box-prompt pilot (P1-9)

Kvasir-SEG test split (200 images) × SAM ViT-B, SAM ViT-H, MedSAM × 24 prompt configurations (box jitter 6 levels × 3 seeds + box scale 6 levels). Acceptance: each model has 200 × 24 = 4800 rows; the number of `invalid` rows is reported (expected ≈ 0).

**Resumable**: if Colab disconnects, just *Run all* again; finished images are skipped and half-written rows are repaired. Needs P1_00 first (P1_01 and P1_02 recommended).

At the end a zip of all results is downloaded. Unzip it into `D:\cs5330\promptshift\results\` and tell Claude Code (P1-9).

**Before you start** (once per Google account):
1. (Only if the repo is private again) Colab left bar → 🔑 *Secrets* → add `GH_TOKEN` (read access
   to `geekRoy0126/promptshift`), switch on *Notebook access*. Never paste a token into a cell.
2. *Runtime → Change runtime type → T4 GPU*.
3. *Runtime → Run all*. Allow Google Drive access when asked.

All data, weights and results go to `MyDrive/promptshift/`, so a disconnect loses nothing:
rerun the notebook and finished steps are skipped.

In [ ]:
# Parameters (edit before Run all if needed)
EXP_ID = "p1_kvasir_pilot"
MODELS = None  # None = all models in the config
MAX_IMAGES = None  # None = whole test split; a small number = quick smoke run

In [ ]:
# SETUP: identical in every notebook (generated by scripts/build_notebooks.py).
import base64
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/geekRoy0126/promptshift.git"
REF = "main"  # branch, tag or commit to run

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    REPO = Path("/content/promptshift")
    try:  # optional: the repo is public; GH_TOKEN is only needed if it is made private again
        _token = userdata.get("GH_TOKEN")
    except Exception:  # secret missing or notebook access not granted
        _token = ""
    _auth = base64.b64encode(f"x-access-token:{_token}".encode()).decode() if _token else ""

    def _git(*args, auth=False):
        pre = ["-c", f"http.extraHeader=Authorization: Basic {_auth}"] if auth and _auth else []
        r = subprocess.run(["git", *pre, "-C", str(REPO), *args], capture_output=True, text=True)
        if r.returncode:
            msg = r.stderr or r.stdout
            for s in (_token, _auth):
                msg = msg.replace(s, "***") if s else msg
            raise RuntimeError(f"git {args[0]} failed: {msg}")

    REPO.mkdir(parents=True, exist_ok=True)
    if not (REPO / ".git").is_dir():
        _git("init", "-q")
    _git("fetch", "-q", "--depth", "1", REPO_URL, REF, auth=True)
    _git("reset", "-q", "--hard", "FETCH_HEAD")  # throwaway clone; all outputs live on Drive
    del _token, _auth
    _pip = [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)]
    r = subprocess.run(_pip, capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError("pip install -e failed (log above)")
else:  # local CPU run: use the checkout this notebook lives in
    REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())

os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
os.environ["PYTHONUNBUFFERED"] = "1"
# scripts must import this checkout's code, whatever else is installed
os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO / "src"), os.environ.get("PYTHONPATH", "")])


def run(*args, check=True):
    """Run a repo script with live output; stop the notebook if it fails. Returns the output."""
    cmd = [sys.executable, *map(str, args)]
    print("$ python", " ".join(cmd[1:]), flush=True)
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = []
    while chunk := p.stdout.read1(4096):
        text = chunk.decode(errors="replace")
        out.append(text)
        sys.stdout.write(text)
        sys.stdout.flush()
    if p.wait() and check:
        raise RuntimeError(f"exit code {p.returncode}: python {' '.join(cmd[1:])}")
    return "".join(out)


def check_gpu():
    import torch

    if IN_COLAB and not torch.cuda.is_available():
        raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU, then Run all again.")
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")


def git_out(*args):
    r = subprocess.run(["git", "-C", str(REPO), *args], capture_output=True, text=True)
    return r.stdout.strip()


SHA = git_out("rev-parse", "--short", "HEAD")
print(f"repo {REPO} @ {SHA} (Colab: {IN_COLAB})")

In [ ]:
check_gpu()
dirty = git_out("status", "--porcelain", "--untracked-files=no")
if dirty:
    print("WARNING: tracked files changed in the clone; git_sha will end in -dirty:")
    print(dirty)

## 1. Run the pilot

In [ ]:
args = ["scripts/run_experiment.py", "--config", "configs/p1_kvasir_pilot.yaml",
        "--exp-id", EXP_ID]
if MODELS:
    args += ["--models", *MODELS]
if MAX_IMAGES:
    args += ["--max-images", MAX_IMAGES]
run(*args)

## 2. Run integrity (acceptance)

In [ ]:
import json

import pandas as pd
from IPython.display import display

from promptshift.paths import get_paths

paths = get_paths()
exp_dir = paths.raw_results / EXP_ID
rows = []
for f in sorted(exp_dir.glob("*_summary.json")):
    s = json.loads(f.read_text())
    rows.append({
        "model": f.name.removesuffix("_summary.json"),
        "rows": s["rows_in_csv"], "expected": s["expected_rows"],
        "invalid rows": s["invalid_rows_in_csv"], "errors": s["errors"],
        "complete": s["complete"],
    })
integrity = pd.DataFrame(rows)
display(integrity)
assert integrity["complete"].all() and (integrity["errors"] == 0).all()

## 3. Report tables (preliminary)

In [ ]:
from IPython.display import Markdown

run("scripts/make_report_tables.py", "--exp-id", EXP_ID)
for t in ["p1_pilot_integrity", "p1_pilot_box_jitter", "p1_pilot_box_scale",
          "p1_pilot_delta"]:
    display(Markdown(f"**{t}**\n\n" + (REPO / "reports" / "tables" / f"{t}.md").read_text()))

## 4. Qualitative examples
Runs the models on 3 test images (box jitter 0 / 10 / 30 / 50 %) and saves `MyDrive/promptshift/results/figures/p1_kvasir_examples.png`. The other figures are made from the CSVs by `scripts/make_figures.py` after the export.

In [ ]:
from IPython.display import Image

run("scripts/make_figures.py", "--examples-only", "--exp-id", EXP_ID,
    "--out-dir", paths.results / "figures", *(["--models", *MODELS] if MODELS else []))
display(Image(filename=str(paths.results / "figures" / "p1_kvasir_examples.png")))

## 5. Export
Zips `raw/<EXP_ID>/`, `parity/`, `summary/`, `figures/`, `colab_check/` and `environment_colab.json` from `MyDrive/promptshift/results/` (paths inside the zip are relative to `results/`). A copy stays in `MyDrive/promptshift/export/`.

In [ ]:
import zipfile

export = paths.root / "export"
export.mkdir(exist_ok=True)
zip_path = export / f"p1_03_{EXP_ID}_{SHA}.zip"
parts = [exp_dir, *(paths.results / p for p in
         ("parity", "summary", "figures", "colab_check", "environment_colab.json"))]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for part in parts:
        files = [part] if part.is_file() else sorted(part.rglob("*"))
        for f in files:
            if f.is_file():
                z.write(f, f.relative_to(paths.results))
print(f"wrote {zip_path} ({zip_path.stat().st_size / 2**20:.1f} MB)")
if IN_COLAB:
    from google.colab import files

    files.download(str(zip_path))
print("\nP1_03 done. Unzip it into the repo folder results/ and tell Claude.")